In [1]:
from openai import OpenAI
import requests

client = OpenAI()

In [2]:
BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

def get_popular_movies():

    response = requests.get(f"{BASE_URL}/movies")

    return response.json()

def get_movie_details(id):

    response = requests.get(f"{BASE_URL}/movies/{id}")

    return response.json()

def get_movie_credits(id):

    response = requests.get(f"{BASE_URL}/movies/{id}/credits")

    return response.json()

In [3]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get the list of popular movies",
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get details about a movie using its movie ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get cast and crew information using a movie ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [4]:
SYSTEM_PROMPT = """
You are a movie expert.

You can use these functions:

- get_popular_movies: returns popular movies
- get_movie_details: returns movie details by movie ID
- get_movie_credits: returns cast and crew information by movie ID

Choose the correct function based on the user's request.
"""

In [8]:
def ask_agent(question):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        tools=tools,
    )

    tool_call = response.choices[0].message.tool_calls[0]

    print("Function:")
    print(tool_call.function.name)

    print("\nArguments:")
    print(tool_call.function.arguments)

In [9]:
ask_agent("지금 인기 있는 영화가 무엇인지 알려줘")

Function:
get_popular_movies

Arguments:
{}


In [10]:
ask_agent("movie ID 550에 해당하는 영화가 무엇인지 알려줘")

Function:
get_movie_details

Arguments:
{"id":550}


In [11]:
ask_agent("movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘")

Function:
get_movie_credits

Arguments:
{"id":550}
